In [0]:
prd_cat_sum_df = spark.table('ORDERS_SALES.GOLD.PRODUCT_CATEGORY_SUMMARY')
prd_cat_sum_df.limit(1).display()

In [0]:
prd_cat_enriched_df = spark.table('ORDERS_SALES.SILVER.PRODUCT_CATEGORY_ENRICHED')
prd_cat_enriched_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

prd_cat_stat_df = prd_cat_sum_df.join(
    prd_cat_enriched_df,
    prd_cat_sum_df["product_category_name"] == prd_cat_enriched_df["product_category_name"],
    "left"
).select(
    prd_cat_sum_df["product_category_name"].alias('category_pt'),
    coalesce(prd_cat_enriched_df['product_category_name'], prd_cat_sum_df["product_category_name"]).alias('category_en'),
    initcap(regexp_replace(coalesce(prd_cat_enriched_df['product_category_name'], prd_cat_sum_df["product_category_name"]), '_', ' ')).alias('category_en_display'),
    'avg_weight_gm', 'avg_length_cm', 'avg_height_cm', 'avg_width_cm', 'avg_volume', 'avg_density'
)

prd_cat_stat_df.limit(1).display()

#### LOAD INTO `ORDERS_SALES.GOLD.PRODUCT_CATEGORY_STATS`

In [0]:
prd_cat_stat_df.write.format('delta').mode('overwrite').saveAsTable('ORDERS_SALES.GOLD.PRODUCT_CATEGORY_STATS')